In [ ]:
#DEFINE GENERATOR AND DISCRIMINATOR
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#define generator
class Generator(nn.Module):
    def __init__(self):
        super(Generator,self).__init__()
        self.fc1=nn.Linear(100,256)
        self.fc2=nn.Linear(256,512)
        self.fc3=nn.Linear(512,1024)
        self.fc4=nn.Linear(1024,28*28*1)#eg for MNIST 28X28 images
    def forward(self,z):
        x=torch.relu(self.fc1(z))
        x=torch.relu(self.fc2(x))
        x=torch.relu(self.fc3(x))
        x=torch.tanh(self.fc4(x))
        return x.view(-1,1,28,28)
#define discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator,self).__init__()
        self.fc1=nn.Linear(28*28,1024)
        self.fc2=nn.Linear(1024,512)
        self.fc3=nn.Linear(512,256)
        self.fc4=nn.Linear(256,1)
    def forward(self,x):
        x=x.view(-1,28*28)
        x=F.leaky_relu(self.fc1(x),0.2)
        x=F.leaky_relu(self.fc2(x),0.2)
        x=F.leaky_relu(self.fc3(x),0.2)
        x=torch.sigmoid(self.fc4(x))
        return x

In [ ]:
#LOAD MNIST DATA
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
batch_size=64
transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize([0.5],[0.5])])
train_dataset=datasets.MNIST(root="./data",train=True,download=True,transform=transform)
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

#TRAIN GAN
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator=Generator().to(device)
discriminator=Discriminator().to(device)
#optimizers
lr=0.0002
optimizer_g=optim.Adam(generator.parameters(),lr=lr,betas=(0.5,0.999))
optimizer_d=optim.Adam(discriminator.parameters(),lr=lr,betas=(0.5,0.999))
#loss function
criterion=nn.BCELoss()
#training loop
for epoch in range(100):
    for real_images,_ in train_loader:
        real_images=real_images.to(device)
        real_labels=torch.ones(real_images.size(0),1).to(device)
        fake_labels=torch.zeros(real_images.size(0),1).to(device)
        #train discriminator
        optimizer_d.zero_grad()
        output=discriminator(real_images)
        loss_real=criterion(output,real_labels)
        z=torch.randn(real_images.size(0),100).to(device)
        fake_images=generator(z)
        output=discriminator(fake_images.detach())
        loss_fake=criterion(output,fake_labels)
        loss_d=loss_real+loss_fake
        loss_d.backward()
        optimizer_d.step()
        #train generator
        optimizer_g.zero_grad()
        output=discriminator(fake_images)
        loss_g=criterion(output,real_labels) #generator wants discriminator to think fakes are real
        loss_g.backward()
        optimizer_g.step()
    real_acc=(discriminator(real_images)>=0.5).float().mean().item()*100
    fake_acc=(discriminator(fake_images.detach())<0.5).float().mean().item()*100
    print(f"Epoch [{epoch+1}/100], Loss D: {loss_d.item():.4f}, Loss G: {loss_g.item():.4f}, Real Acc: {real_acc:.1f}%, Fake Acc: {fake_acc:.1f}%")

In [ ]:
#SAVE GENERATOR
torch.save(generator.state_dict(),"generator.pth")
print("Generator saved to generator.pth")

In [2]:
!wget "https://raw.githubusercontent.com/econsmaestro/building-an-image-generator/  master/train_wgangp_colab.py"


--2026-05-28 11:37:15--  https://raw.githubusercontent.com/econsmaestro/building-an-image-generator/%20%20master/train_wgangp_colab.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-28 11:37:15 ERROR 404: Not Found.



In [ ]:
!wget https://raw.githubusercontent.com/econsmaestro/building-an-image-generator/refs/heads/master/train_wgangp_colab.py
